# Embedding a Knowledge Assistant in Microsoft Teams

This notebook walks through the production-ready way to put a Databricks Knowledge Assistant in front of Teams users: deploy a small Streamlit chat UI as a Databricks App, then add it as a tab in a Teams channel.

### Why a Databricks App and not just the Knowledge Assistant URL directly

The Knowledge Assistant agent in Agent Bricks has a built-in chat URL inside Databricks, and that URL can technically be added as a Teams Website tab. That path is fine for a quick internal demo, but it has three real limitations:

- Every Teams user needs a Databricks workspace login. First-time users get a Databricks SSO prompt inside the Teams tab.
- The Databricks workspace navigation appears inside the Teams tab. The experience looks like "Databricks running inside Teams" instead of a purpose-built chatbot.
- There is no branding control. You cannot change colors, add a logo, or customize the empty-state message.

The Databricks App approach removes all three of those limitations. The App runs as a service principal so end users do not need individual Databricks accounts. The UI is a clean Streamlit chat panel with no Databricks chrome. Branding is fully customizable. This is the right path for any production-facing rollout.

Time required: about 30-45 minutes. Most of the work is click-through in Databricks and Microsoft Teams.

## Prerequisites

- A deployed Knowledge Assistant agent in Databricks Agent Bricks. The agent must have a serving endpoint name available (visible on the agent's detail page).
- A Databricks workspace with Apps enabled.
- A Microsoft Teams workspace where you have permission to add tabs to channels.

## Placeholders used in this notebook

| Placeholder | Description |
|---|---|
| `<KA_ENDPOINT_NAME>` | The Knowledge Assistant serving endpoint name from Agent Bricks |
| `<WORKSPACE_URL>` | Your Databricks workspace URL (e.g., `https://example.cloud.databricks.com`) |
| `<APP_NAME>` | Name you choose for the Databricks App (e.g., `ka-chat`) |

## Step 1: Verify the Knowledge Assistant endpoint is queryable

Before building any UI, confirm the endpoint actually responds. This avoids debugging Teams or App configuration when the real problem is on the Databricks serving side.

Run the cell below from any Databricks notebook with access to your Databricks PAT. A successful response confirms the endpoint is healthy and the conversation API is working as expected.

In [0]:
import os
import requests

WORKSPACE_URL = '<WORKSPACE_URL>'
KA_ENDPOINT_NAME = '<KA_ENDPOINT_NAME>'
TOKEN = dbutils.secrets.get(scope='<scope>', key='databricks_pat')

response = requests.post(
    f'{WORKSPACE_URL}/serving-endpoints/{KA_ENDPOINT_NAME}/invocations',
    headers={'Authorization': f'Bearer {TOKEN}', 'Content-Type': 'application/json'},
    json={
        'messages': [
            {'role': 'user', 'content': 'What is this assistant for?'}
        ]
    },
    timeout=60,
)

print('Status:', response.status_code)
print('Body:', response.json())

## Step 2: Build the chat UI

The chat UI is a small Streamlit app whose only job is to talk to the Knowledge Assistant serving endpoint. The code below:

- Streams responses token-by-token using `st.write_stream` so users see text appear as it is generated. No waiting on the full response.
- Reuses a single HTTP connection across requests for low per-message latency.
- Hides the Streamlit menu, footer, and header so the UI looks clean inside the Teams tab.
- Handles errors gracefully with user-friendly messages instead of stack traces.
- Maintains conversation history within a session.

Create a folder anywhere in your Databricks workspace. Save the Python code below as `app.py` in that folder. The next cell covers the two supporting files that go alongside it.

In [0]:
# app.py
# Minimal, fast, clean chat UI that calls a Knowledge Assistant serving endpoint.

import json
import os

import requests
import streamlit as st

WORKSPACE_URL    = os.environ['DATABRICKS_HOST'].rstrip('/')
KA_ENDPOINT_NAME = os.environ['KA_ENDPOINT_NAME']
TOKEN            = os.environ['DATABRICKS_TOKEN']

st.set_page_config(
    page_title='Knowledge Assistant',
    layout='centered',
    initial_sidebar_state='collapsed',
)

# Hide Streamlit chrome for a clean Teams-friendly look
st.markdown(
    '''
    <style>
      #MainMenu, footer, header { visibility: hidden; }
      .block-container { padding-top: 1rem; padding-bottom: 1rem; }
      [data-testid="stChatMessage"] { padding: 0.5rem 1rem; }
    </style>
    ''',
    unsafe_allow_html=True,
)

# Reused HTTP session for connection pooling (lower latency per turn)
if 'http' not in st.session_state:
    sess = requests.Session()
    sess.headers.update({
        'Authorization': f'Bearer {TOKEN}',
        'Content-Type': 'application/json',
    })
    st.session_state.http = sess

if 'messages' not in st.session_state:
    st.session_state.messages = []

st.title('Knowledge Assistant')

# Render conversation history
for msg in st.session_state.messages:
    with st.chat_message(msg['role']):
        st.markdown(msg['content'])


def stream_response(messages):
    """Stream tokens from the Knowledge Assistant serving endpoint."""
    url = f'{WORKSPACE_URL}/serving-endpoints/{KA_ENDPOINT_NAME}/invocations'
    payload = {'messages': messages, 'stream': True}

    with st.session_state.http.post(url, json=payload, stream=True, timeout=120) as resp:
        if resp.status_code != 200:
            yield f'Error from Knowledge Assistant: HTTP {resp.status_code}'
            return

        for line in resp.iter_lines(decode_unicode=True):
            if not line or not line.startswith('data: '):
                continue
            data_str = line[6:]
            if data_str == '[DONE]':
                break
            try:
                chunk = json.loads(data_str)
                delta = chunk.get('choices', [{}])[0].get('delta', {}).get('content', '')
                if delta:
                    yield delta
            except json.JSONDecodeError:
                continue


# Handle new user input
if prompt := st.chat_input('Ask a question'):
    st.session_state.messages.append({'role': 'user', 'content': prompt})
    with st.chat_message('user'):
        st.markdown(prompt)

    with st.chat_message('assistant'):
        try:
            answer = st.write_stream(stream_response(st.session_state.messages))
        except requests.exceptions.Timeout:
            answer = 'The request took too long. Please try a more focused question.'
            st.error(answer)
        except Exception as exc:
            answer = f'Something went wrong: {str(exc)[:200]}'
            st.error(answer)
        st.session_state.messages.append({'role': 'assistant', 'content': answer})

### Supporting files

Save the following two files alongside `app.py` in the same folder.

**`app.yaml`**

```yaml
command: ['streamlit', 'run', 'app.py', '--server.port', '8080', '--server.address', '0.0.0.0', '--server.headless', 'true']

env:
  - name: KA_ENDPOINT_NAME
    value: '<KA_ENDPOINT_NAME>'
```

**`requirements.txt`**

```text
streamlit==1.39.0
requests==2.32.3
```

Replace `<KA_ENDPOINT_NAME>` in `app.yaml` with your actual Knowledge Assistant serving endpoint name.

## Step 3: Deploy the App

1. In Databricks, click **Compute** > **Apps** in the left sidebar.
2. Click **Create app**.
3. Name it (e.g., `ka-chat`). Click **Create**.
4. Choose the workspace folder where your three files live (`app.py`, `app.yaml`, `requirements.txt`).
5. Click **Deploy**. Wait for the status to show "Running" (1-2 minutes the first time).

Once running, the app shows a URL at the top of its page:

```
https://<app-name>-<workspace-id>.<region>.databricksapps.com
```

Copy this URL. Open it in a new browser tab and verify the chat UI loads and answers a test question.

### Performance: keep the app warm

By default, Databricks Apps scale to zero when idle. The first request after idle hits a cold start of 10-30 seconds. For a production user-facing chatbot this is too slow.

Two options:

1. Configure the App's compute to stay always-on. In the app's Compute settings, set the minimum instances to 1. Slightly higher cost, no cold starts.
2. Schedule a cheap keep-alive ping every 5 minutes. Set up a Databricks Job that runs `curl https://<your-app-url>/healthz` on a 5-minute cron. Free, but adds a small orchestration dependency.

Pick option 1 for production. Use option 2 only if cost is a critical constraint.

## Step 4: Grant the App permission to call the Knowledge Assistant

The App runs as a service principal. That service principal needs `CAN_QUERY` permission on the Knowledge Assistant serving endpoint.

1. In Databricks, navigate to **Compute** > **Serving**.
2. Find the Knowledge Assistant endpoint (named `<KA_ENDPOINT_NAME>`).
3. Click the endpoint to open it.
4. Click the **Permissions** tab.
5. Click **Grant**. Select the App's service principal (its name will match `<APP_NAME>-app`).
6. Check `CAN_QUERY`. Click **Grant**.

If you skip this step, the App will return HTTP 403 when users ask questions.

## Step 5: Add the App as a Teams tab

1. Open Microsoft Teams.
2. Navigate to the channel where the Knowledge Assistant should live.
3. At the top of the channel, click **+ (Add a tab)**.
4. Search for and select **Website**.
5. Set:
   - Tab name: `Knowledge Assistant`
   - URL: the Databricks App URL from Step 3
6. Click **Save**.

The chat UI now appears as a Teams tab with clean branding, no Databricks chrome, and no per-user Databricks login prompt.

## Step 6: Test the end-user experience

1. Open the Teams channel from a colleague's account.
2. Click the Knowledge Assistant tab.
3. Confirm the chat loads quickly (sub-second, if the App is warm).
4. Ask three to five real questions. Confirm:
   - Responses stream in token-by-token, not as one big block.
   - Citations and source references appear correctly.
   - Conversation history persists across messages within a session.
   - Refreshing the tab clears the conversation (Streamlit session resets).

If responses feel slow, check:

- Is the app scaled to zero? See Step 3's keep-warm options.
- Is the Knowledge Assistant endpoint provisioning enough throughput? Check the endpoint's metrics panel.
- Is the user's network adding latency? Try from a fast network.

## Heavier alternatives, for completeness

The Databricks App + Teams tab pattern above covers the vast majority of use cases. Two more sophisticated alternatives exist if a specific limitation forces it:

- **Native Microsoft Teams Bot (Bot Framework SDK).** Users can `@mention` the bot in any channel or DM, not just inside a specific channel tab. Requires Microsoft Bot Framework SDK, Azure Bot Service registration, and Azure hosting (Functions or App Service). Multi-week development effort. Use when stakeholders demand chat-anywhere instead of chat-in-a-tab.
- **Microsoft Copilot Studio custom skill.** Wraps the Knowledge Assistant endpoint as a Copilot skill, published to Teams via Copilot Studio. Adds Copilot Studio licensing. Use when the organization already invests in Copilot Studio and wants centralized agent governance there.

Both alternatives still point at the same Knowledge Assistant serving endpoint. Only the consumption layer changes.

## References

- [Knowledge Assistant](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/knowledge-assistant)
- [Databricks Apps overview](https://docs.databricks.com/aws/en/dev-tools/databricks-apps/)
- [Streamlit in Databricks Apps](https://docs.databricks.com/aws/en/dev-tools/databricks-apps/get-started/streamlit)
- [Model Serving endpoint permissions](https://docs.databricks.com/aws/en/machine-learning/model-serving/manage-serving-endpoints)
- [Adding Website tabs in Microsoft Teams](https://learn.microsoft.com/en-us/microsoftteams/manage-website-tab)